In [1]:
import pandas as pd
import numpy as np

from collections import defaultdict
from itertools import combinations
from scipy.sparse import csr_matrix

In [2]:
#ファイル読み込み
df = pd.read_csv('../000.data/train/train_A.tsv', sep="\t")

In [3]:
# event_typeがカートに入れるのレコードを除去 + event_typeが購入だけど広告経由でないのレコードを除去
df_filtered = df[(df["event_type"] != 0) & ~((df["event_type"] == 3) & (df["ad"] != 1))]

In [4]:
# ユーザーごとの購入商品リストを作成
user_product = df_filtered.groupby("user_id")["product_id"].apply(list)

In [5]:
# 共起カウント用の辞書
co_occurrence = defaultdict(int)

In [6]:
# 各ユーザーごとに、購入商品のペアをカウント
for products in user_product:
    for p1, p2 in combinations(products, 2):
        co_occurrence[(p1, p2)] += 1
        co_occurrence[(p2, p1)] += 1  # 対称性を持たせる

In [7]:
# ユニークな商品リスト
unique_products = sorted(set(df_filtered["product_id"]))
product_index = {p: i for i, p in enumerate(unique_products)}

In [8]:
# 共起行列の作成
n_products = len(unique_products)
co_matrix = np.zeros((n_products, n_products), dtype=int)

for (p1, p2), count in co_occurrence.items():
    i, j = product_index[p1], product_index[p2]
    co_matrix[i, j] = count
    co_matrix[j, i] = count  # 対称性を持たせる

In [9]:
# CSRスパース行列に変換
co_matrix_sparse = csr_matrix(co_matrix)

In [10]:
# DataFrame化（デバッグ用）
co_matrix_df = pd.DataFrame(co_matrix, index=unique_products, columns=unique_products)
print(co_matrix_df)

            00000000_a  00000001_a  00000002_a  00000003_a  00000004_a  \
00000000_a          32           2           0           5           1   
00000001_a           2         244           0           3           1   
00000002_a           0           0           0           0           0   
00000003_a           5           3           0        2162           1   
00000004_a           1           1           0           1           0   
...                ...         ...         ...         ...         ...   
00014118_a           1           1           0           1           1   
00014119_a           0           0           0           0           0   
00014120_a           2           6           0          31           1   
00014121_a           0           0           0           4           0   
00014122_a           1           1           0           1           1   

            00000005_a  00000006_a  00000007_a  00000008_a  00000009_a  ...  \
00000000_a           5          

In [11]:
# Jaccard類似度を計算して上位22件を推薦
def jaccard_similarity(product_id, co_matrix_df, top_n=22):
    if product_id not in co_matrix_df.index:
        return []
    
    co_counts = co_matrix_df.loc[product_id]
    total_counts = co_matrix_df.sum(axis=1)
    jaccard_scores = co_counts / (total_counts + total_counts[product_id] - co_counts)
    jaccard_scores = jaccard_scores.drop(product_id).sort_values(ascending=False)
    return jaccard_scores.head(top_n).index.tolist()

In [12]:
# テスト
target_product = "00009250_a"
candidates = jaccard_similarity(target_product, co_matrix_df, top_n=22)
print("候補商品:", candidates)

候補商品: ['00008251_a', '00003009_a', '00014068_a', '00008525_a', '00010006_a', '00010537_a', '00010972_a', '00011337_a', '00003069_a', '00003524_a', '00000797_a', '00004433_a', '00007552_a', '00003316_a', '00011513_a', '00011250_a', '00003679_a', '00010698_a', '00008263_a', '00008658_a', '00000953_a', '00010017_a']
